In [6]:
import retrieve_dspy

listwise_reranker = retrieve_dspy.ListwiseReranker(
    collection_name="FreshstackLangchain",
    target_property_name="docs_text",
    diverse_ranker=True,
    retrieved_k=50,
    reranked_k=20
)

listwise_reranker("How can I use Weaviate with LangChain?")

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/pydantic/main.py:453: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 6: Expected `Message` - serialized value may not be as expected [input_value=Message(content='[[ ## re...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


Prediction(
    final_answer='',
    sources=[Source(object_id='5163bd72-2249-4fa0-9ac4-7ba904a7f4e4'), Source(object_id='1cd71614-2aff-4961-a431-a05d5c37f25c'), Source(object_id='4cf9a2cb-e9ec-4883-8075-05054f38f8a6'), Source(object_id='b980876d-0521-4440-abf5-ea2b64dc96ff'), Source(object_id='9608f261-9b02-4a9e-ab23-54b3b4c704f8'), Source(object_id='eabdeb84-50fd-43c0-8711-a0ec0a3d34b2'), Source(object_id='ca6fcd08-ef78-4118-b132-757f71cfd1ca'), Source(object_id='a878bad7-1e66-427d-9672-aa96164bb41b'), Source(object_id='542bf62c-7fa7-414d-88b5-2485d11012b1')],
    searches=['How can I use Weaviate with LangChain?'],
    aggregations=None,
    usage={}
)

In [14]:
from retrieve_dspy.metrics import create_metric
from retrieve_dspy.datasets.in_memory import load_queries_in_memory

trainset, testset = load_queries_in_memory(
    dataset_name="freshstack-langchain",
    train_samples=10,
    test_samples=10
)

metric = create_metric(
    metric_type="coverage",
    dataset_name="freshstack-langchain"
)

evaluator = retrieve_dspy.utils.get_evaluator(
    testset=testset,
    metric=metric
)

In [16]:
# Add this cell to debug
print("Metric type:", type(metric))
print("Metric contents:", metric)

Metric type: <class 'function'>
Metric contents: <function create_coverage_metric.<locals>.coverage_metric at 0x330629900>


In [19]:
dspy_evaluator_kwargs = {
    "num_threads": 4
}

evaluator(listwise_reranker, **dspy_evaluator_kwargs)

  0%|          | 0/10 [00:00<?, ?it/s]Coverage@100 evaluation:
Total nuggets: 2
Covered nuggets: 2
Nugget 1: Covered
Nugget 2: Covered
Coverage@100: 2/2 = 1.00
Average Metric: 1.00 / 1 (100.0%):  10%|█         | 1/10 [00:02<00:23,  2.59s/it]Coverage@100 evaluation:
Total nuggets: 3
Covered nuggets: 0
Nugget 1: Not covered
Nugget 2: Not covered
Nugget 3: Not covered
Coverage@100: 0/3 = 0.00
Average Metric: 1.00 / 2 (50.0%):  20%|██        | 2/10 [00:03<00:13,  1.63s/it] Coverage@100 evaluation:
Total nuggets: 3
Covered nuggets: 3
Nugget 1: Covered
Nugget 2: Covered
Nugget 3: Covered
Coverage@100: 3/3 = 1.00
Average Metric: 2.00 / 3 (66.7%):  30%|███       | 3/10 [00:04<00:09,  1.32s/it]Coverage@100 evaluation:
Total nuggets: 3
Covered nuggets: 1
Nugget 1: Not covered
Nugget 2: Not covered
Nugget 3: Covered
Coverage@100: 1/3 = 0.33
Average Metric: 2.33 / 4 (58.3%):  40%|████      | 4/10 [00:06<00:10,  1.73s/it]Coverage@100 evaluation:
Total nuggets: 4
Covered nuggets: 2
Nugget 1: Covered

2025/08/06 17:13:24 INFO dspy.evaluate.evaluate: Average Metric: 4.916666666666667 / 10 (49.2%)


sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x3301ba440>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x339d7e500>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x339bde260>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x33b082680>


49.17

In [ ]:
import dspy

optimizer = dspy.MIPROv2(
    metric=metric,
    auto="heavy",
    verbose=True
)

optimized_listwise_reranker = optimizer.compile(
    listwise_reranker,
    trainset=trainset,
    requires_permission_to_run=False
)

In [21]:
print("MIPRO run is finished!")

MIPRO run is finished!


In [22]:
optimized_listwise_reranker.save("mipro_optimized_listwise_reranker.json")

In [23]:
evaluator(optimized_listwise_reranker, **dspy_evaluator_kwargs)

  0%|          | 0/10 [00:00<?, ?it/s]Coverage@100 evaluation:
Total nuggets: 2
Covered nuggets: 2
Nugget 1: Covered
Nugget 2: Covered
Coverage@100: 2/2 = 1.00
Average Metric: 1.00 / 1 (100.0%):  10%|█         | 1/10 [00:04<00:39,  4.40s/it]Coverage@100 evaluation:
Total nuggets: 3
Covered nuggets: 3
Nugget 1: Covered
Nugget 2: Covered
Nugget 3: Covered
Coverage@100: 3/3 = 1.00
Average Metric: 2.00 / 2 (100.0%):  20%|██        | 2/10 [00:04<00:15,  1.89s/it]Coverage@100 evaluation:
Total nuggets: 3
Covered nuggets: 1
Nugget 1: Not covered
Nugget 2: Not covered
Nugget 3: Covered
Coverage@100: 1/3 = 0.33
Average Metric: 2.33 / 3 (77.8%):  30%|███       | 3/10 [00:05<00:10,  1.52s/it] Coverage@100 evaluation:
Total nuggets: 3
Covered nuggets: 3
Nugget 1: Covered
Nugget 2: Covered
Nugget 3: Covered
Coverage@100: 3/3 = 1.00
Average Metric: 3.33 / 4 (83.3%):  30%|███       | 3/10 [00:05<00:10,  1.52s/it]Coverage@100 evaluation:
Total nuggets: 4
Covered nuggets: 4
Nugget 1: Covered
Nugget 2: 

2025/08/06 17:22:46 INFO dspy.evaluate.evaluate: Average Metric: 6.083333333333333 / 10 (60.8%)


sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x32d1e0400>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x32d1e1600>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x346f78e20>


60.83

sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x341f0b160>
